## Projet Santé Mentale des Adolescents

In [42]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.2 s3fs==2026.3.0 -q

Note: you may need to restart the kernel to use updated packages.


## Importation des packages nécessaires

In [43]:
import pandas as pd
import os
import openpyxl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(openpyxl.__version__)


3.1.3


### Importation des données - Santé Mentale

Après l'importation, on inspecte les types de données présents.

In [44]:
df = pd.read_csv('https://minio.lab.sspcloud.fr/nerojeni10/DATA_PROJET_SMA/Teen_Mental_Health_Dataset.csv')

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   str    
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   str    
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_level  1200 non-null   str    
 9   stress_level              1200 non-null   int64  
 10  anxiety_level             1200 non-null   int64  
 11  addiction_level           1200 non-null   int64  
 12  depression_label          1200 non-null   int64  
dtypes: float64(5), int64(5), str(3)
memory usage: 140.4 KB


,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0


### Inspecter la présence des valeurs manquantes

In [45]:
df.isnull().sum()

age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64

***Il n'y a aucune valeur manquante dans ce jeux de données***

## Remplissage du fichier

Pour remplir le fichier, on allons créer plusieurs feuilles composées des données nécessaires à la création des indicateurs

In [47]:
path_file = "../template/Projet_ODD_SIVARAJAH.xlsx"

# Recréer un fichier propre sans feuille parasite
try:
    wb = load_workbook(path_file)
except Exception:
    wb = Workbook()


# Créer un vrai fichier Excel vide si inexistant
if not os.path.exists(path_file):
    wb = Workbook()
    wb.save(path_file)


# Ajouter la feuille DATA
with pd.ExcelWriter(path_file, mode="a", if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name='DATA', index=False)

print("Feuilles présentes :", load_workbook(path_file).sheetnames)

Feuilles présentes : ['Sheet', 'DATA']


## Création des indicateurs

In [49]:
# Chargement du fichier en mémoire
wb = load_workbook(path_file)


# Supprimer la feuille vide par défaut si elle existe
if "Sheet" in wb.sheetnames:
    del wb["Sheet"]

wb.save(path_file)

# Créer la feuille Indicateurs si elle n'existe pas
if "Indicateurs" not in wb.sheetnames:
    ws = wb.create_sheet("Indicateurs")
else:
    ws = wb["Indicateurs"]

# Ajout des formules
# 1. Nombre de filles dépressives
ws['A1'] = "Nombre de filles dépressives"
ws['B1'] = '=COUNTIFS(DATA!M:M,1,DATA!B:B,"female")'

# 2. Nombre de garçons dépressifs
ws['A2'] = "Nombre de garçons dépressifs"
ws['B2'] = '=COUNTIFS(DATA!M:M,1,DATA!B:B,"male")'

# 3. Niveau d'addiction moyen chez les filles
ws['A3'] = "Niveau d'addiction moyen chez les filles"
ws['B3'] = '=AVERAGEIF(DATA!B:B,"female",DATA!L:L)'

# 4. Niveau d'addiction moyen chez les garçons
ws['A4'] = "Niveau d'addiction moyen chez les garçons"
ws['B4'] = '=AVERAGEIF(DATA!B:B,"male",DATA!L:L)'

# Création de la feuille TCD si elle n'existe pas
if "TCD" not in wb.sheetnames:
    ws_tcd = wb.create_sheet("TCD")
else:
    ws_tcd = wb["TCD"]

# 5. Niveau de dépression selon l'âge
tcd1 = df.groupby(["age"])["depression_label"].sum().reset_index()
tcd1.columns = ["Age", "Nb depressifs"]

start_row = 1
ws_tcd.cell(row=start_row, column=1, value="Age")
ws_tcd.cell(row=start_row, column=2, value="Nb depressifs")
for i, row in tcd1.iterrows():
    ws_tcd.cell(row=start_row+i+1, column=1, value=row["Age"])
    ws_tcd.cell(row=start_row+i+1, column=2, value=row["Nb depressifs"])

# 6. Niveau d'addiction moyen selon l'âge et le genre
tcd2 = df.groupby(["age", "gender"])["addiction_level"].mean().round(2).unstack()
tcd2.columns.name = None
tcd2 = tcd2.reset_index()
tcd2.columns = ["Age", "Addiction moy. Filles", "Addiction moy. Garcons"]

start_row = start_row + len(tcd1) + 3
ws_tcd.cell(row=start_row, column=1, value="Age")
ws_tcd.cell(row=start_row, column=2, value="Addiction moy. Filles")
ws_tcd.cell(row=start_row, column=3, value="Addiction moy. Garcons")
for i, row in tcd2.iterrows():
    ws_tcd.cell(row=start_row+i+1, column=1, value=row["Age"])
    ws_tcd.cell(row=start_row+i+1, column=2, value=row["Addiction moy. Filles"])
    ws_tcd.cell(row=start_row+i+1, column=3, value=row["Addiction moy. Garcons"])

# 7. Nombre de depressions selon le temps de sommeil et l'interaction sociale
df["sleep_group"] = pd.cut(df["sleep_hours"],
                            bins=[0, 5, 6, 7, 8, 12],
                            labels=["<5h", "5-6h", "6-7h", "7-8h", ">8h"])
tcd3 = df.groupby(["social_interaction_level", "sleep_group"])["depression_label"].sum().unstack()
tcd3.columns.name = None
tcd3 = tcd3.reset_index()
tcd3.columns = ["Interaction sociale", "<5h", "5-6h", "6-7h", "7-8h", ">8h"]

start_row = start_row + len(tcd2) + 3
for col_idx, header in enumerate(tcd3.columns, start=1):
    ws_tcd.cell(row=start_row, column=col_idx, value=header)
for i, row in tcd3.iterrows():
    for col_idx, val in enumerate(row, start=1):
        ws_tcd.cell(row=start_row+i+1, column=col_idx, value=val)

# 8. Repartition niveau addiction selon l'age (indicateurs boite a moustaches)
summary_box = df.groupby("age")["addiction_level"].agg(
    Q1      = lambda x: x.quantile(0.25),
    Mediane = lambda x: x.quantile(0.50),
    Q3      = lambda x: x.quantile(0.75),
    Min     = "min",
    Max     = "max",
    Moyenne = "mean",
    IQR     = lambda x: x.quantile(0.75) - x.quantile(0.25)
).round(2).reset_index()
summary_box.columns = ["Age", "Q1", "Mediane", "Q3", "Min", "Max", "Moyenne", "IQR"]

start_row = start_row + len(tcd3) + 3
for col_idx, header in enumerate(summary_box.columns, start=1):
    ws_tcd.cell(row=start_row, column=col_idx, value=header)
for i, row in summary_box.iterrows():
    for col_idx, val in enumerate(row, start=1):
        ws_tcd.cell(row=start_row+i+1, column=col_idx, value=val)


# Encoder les variables catégorielles en numérique
df["gender_num"] = df["gender"].map({"male": 0, "female": 1})
df["social_num"] = df["social_interaction_level"].map({"low": 0, "medium": 1, "high": 2})

# Sélectionner les colonnes numériques
cols_corr = ["age", "sleep_hours", "daily_social_media_hours",
             "academic_performance", "physical_activity",
             "social_num", "stress_level", "anxiety_level",
             "addiction_level", "depression_label"]

# Matrice de corrélation
corr = df[cols_corr].corr().round(2)

corr_reset = corr.reset_index()
corr_reset.columns = ["Variable"] + cols_corr

wb = load_workbook(path_file)

if "Correlations" not in wb.sheetnames:
    ws_corr = wb.create_sheet("Correlations")
else:
    ws_corr = wb["Correlations"]

# Headers
for col_idx, header in enumerate(corr_reset.columns, start=1):
    ws_corr.cell(row=1, column=col_idx, value=header)

# Données
for i, row in corr_reset.iterrows():
    for col_idx, val in enumerate(row, start=1):
        ws_corr.cell(row=i+2, column=col_idx, value=val)


# Sauvegarde du fichier
wb.save(path_file)
wb.close()
print("Feuilles presentes :", load_workbook(path_file).sheetnames)

Feuilles presentes : ['DATA', 'Indicateurs', 'TCD', 'Correlations']
